Project 1: Data Analysis Using Pandas (NYC Taxi Revenue Analysis)
The Real-World Scenario: You are hired as a Data Analysts for the NYC Transport Authority. You are handed a dataset of taxi trips containing pickup times, distances, fares, and payment methods. The data has missing values (like unrecorded drop-off zones) and raw date strings. Your job is to clean the data, calculate new metrics (like trip duration and fare per mile), and build a summary report showing where the most revenue is generated.

In [ ]:
import numpy as np
import pandas as pd


In [ ]:
#Load real NYC datasets

url1 = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/taxis.csv'

df = pd.read_csv(url1)

print("--NYC Taxi Data--")
print("Totale trips :", df.shape[0])
print(f"Data point : {df.shape[1]}\n")




In [ ]:
# Display the first 10 rows to inspect the structure and sample data
print("First few rows of the dataset")


df.head(10)

In [ ]:
# Generate descriptive statistics for numerical columns (mean, min, max, std, etc.)
print("\nStatistical Summary:")
df.describe()


# Display the summary of the dataset including columns,non-null count,and data types

In [ ]:
print("\nDataset Information:")
df.info()

#Report Missing Data

In [ ]:
print("\n---Missing values count ---")
missing = df.isnull().sum()[df.isnull().sum()>0]
print(missing)


Data Cleaning:  Real data is messy. We will drop rows with missing zones (since we can't guess where a taxi went) and fill missing payment types with the most common method.



In [ ]:
#Data cleaning
df_clean = df.copy ()
# A. Drop rows where we don't know the pickup or dropoff zone (Critical missing data)
df_clean.dropna(subset=['pickup_zone','dropoff_zone'], inplace = True)
print(f"Rows after dropping missing zones : {df_clean.shape[0]}")




In [ ]:
#B. Fill missing 'payment' types with the mode (most frequent payment method)
most_common_payment = df_clean['payment'].mode()[0]
print(f"Most common pyment mathod: {most_common_payment}")
df_clean['payment'] = df_clean['payment'].fillna(most_common_payment)



Invalid Trips

In [ ]:
invalid_trips = df[ (df['distance'] <= 0)]
print(f"Invalid trips : {invalid_trips.shape[0]}")
print(invalid_trips)

In [ ]:
# C. Remove trips with 0 distance or 0 fare (Data entry errors / canceled rides

df_clean = df_clean[(df_clean['distance'] > 0)&(df_clean['fare']>0)]
print(f"Rows remaining after cleaning :{df_clean.shape[0]}")


4: Feature Engineering (The Power of Datetimes) Converting strings to dates and calculating new business metrics.

In [ ]:
df_clean.head()


In [ ]:
# A. Convert 'pickup' and 'dropoff' from text strings into proper Pandas Datetime objects
df_clean['pickup'] = pd.to_datetime(df_clean['pickup'])
df_clean['dropoff'] = pd.to_datetime(df_clean['dropoff'])



In [ ]:
# B. Calculating Trip Duration in Minutes

df_clean['duration_minutes'] = (df_clean['dropoff'] - df_clean['pickup']).dt.total_seconds() / 60
df_clean['duration_minutes'] = df_clean['duration_minutes'].round(2)




In [ ]:
# C. Calculating Fare per Maile
df_clean['fare_per_mile'] = (df_clean['fare'] / df_clean['distance']).round(2)


In [ ]:
# D. Extract the Hour of the day the trip started

df_clean['pickup_hour'] = df_clean['pickup'].dt.hour

display(df_clean[['pickup','duration_minutes','distance','fare','fare_per_mile']].head())



In [ ]:
df_clean.head()

In [ ]:
from numpy._core.defchararray import count
# 5. Extracting Business Insights

#Insight - 1

borough_summary = df_clean.groupby('pickup_zone').agg(
  Totale_Trips = ('pickup','count'),
  Avg_Fare = ('fare','mean'),
  Avg_Tip = ('tip','mean'),

).round(2).sort_values(by='Totale_Trips',ascending=False)

print("---Borough Summary---")
display(borough_summary.head())





In [ ]:
# Insight 2: Tipping behavior based on Payment Method and Taxi Color (Pivot Table)
tip_pivot = pd.pivot_table(
    df_clean,
    values='tip',
    index='payment',
    columns='color',
    aggfunc='mean'
).round(2)

print("\n---Avarage Tip by payment method and taxi color---")
display(tip_pivot)




In [ ]:
df_clean.to_csv('cleaned_taxi_data.csv', index=False)